# Capítulo 5 — Clasificar y tomar una decisión · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch05_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin producir errores. La solución está en `ch05_corrige.ipynb`, en la misma página complementaria.

**Índice.** Datos · Los cálculos de las lecciones 5.1 a 5.5 · Ejercicios 5.1 a 5.6

**Convenciones del notebook.**
- Regla de decisión (ecuación 5.1): un mensaje se marca si su probabilidad estimada es **mayor o igual** que el umbral. Para los doce mensajes, la regla se aplica al valor **sin redondear** `p_exact`, recalculado con la sigmoide; la tabla del libro solo muestra su redondeo a la centésima. En los ejercicios en los que las probabilidades se dan con decimales, esos decimales son los valores que hay que usar.
- Una medida cuyo denominador es cero (por ejemplo, la precisión cuando no se emite ninguna alerta) vale `None` y se muestra como «indefinida».
- El F1 se calcula como $2\,VP / (2\,VP + FP + FN)$ (ecuación 5.8): VP = verdaderos positivos, FP = falsos positivos, FN = falsos negativos, VN = verdaderos negativos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
SEUIL, ORANGE, BLEU, VERT = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # umbral; sigmoide; pérdida; filtro B


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def affiche(v):
    """Convención del notebook: una medida cuyo denominador es cero vale None y se muestra como «indefinida»."""
    return 'indefinida' if v is None else f'{v:.2f}'

## Datos

Los identificadores D5.1, D5.2, D5.E1, D5.E2 y D5.E3 son los del libro y de los archivos CSV.

Los nombres de variables y columnas del código se mantienen en francés: `douze` / `dix` / `quatre` (doce / diez / cuatro mensajes), `etiquette` (etiqueta real: `'spam'` o `'legitime'`, legítimo), `liens` (enlaces), `mots_suspects` (palabras sospechosas), `score` (probabilidad estimada), `calibration` (filtro A), `calibration_b` (filtro B), `tranche_min` / `tranche_max` (límites del tramo), `proba_moyenne` (probabilidad media predicha), `p_filtre_A` / `p_filtre_B` (probabilidades de los filtros A y B), `p_exact` (probabilidad sin redondear), `est_spam` (es spam), `seuil` (umbral), `signalé` (marcado), `perte` (pérdida), `cout` (coste), `rappel` (exhaustividad).

In [ ]:
# Datos del capítulo 5, copiados de los archivos CSV del libro (no modificar)

# D5.1 — doce mensajes: etiqueta real, enlaces, palabras sospechosas, probabilidad estimada REDONDEADA a la centésima
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'etiquette', 'liens', 'mots_suspects', 'score'])

# D5.2 — filtro A: 1000 mensajes por tramo (número de mensajes, spams, probabilidad media predicha)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E1 — filtro B: misma tabla (ejercicio 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E2 — diez mensajes de la asociación (ejercicio 5.2): estos decimales son los valores que hay que usar
dix = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'etiquette', 'score'])

# D5.E3 — cuatro mensajes puntuados por dos filtros (ejercicio 5.3)
quatre = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'etiquette', 'p_filtre_A', 'p_filtre_B'])

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 5.1 a 5.5.

In [ ]:
# Valores exactos de los doce mensajes (ecuaciones 5.3 y 5.4): la regla se aplica a ellos
w, b = np.array([1, 0.5]), -4
X = douze[['liens', 'mots_suspects']].to_numpy()
z = X @ w + b
p_exact = sigmoide(z)
est_spam = (douze['etiquette'] == 'spam').to_numpy()
y = est_spam.astype(int)                 # 1 = spam, 0 = legítimo

# Lección 5.1 — mover el umbral (ecuaciones 5.1 y 5.2)
for seuil in (0.5, 0.25):
    signale = p_exact >= seuil                    # regla: mayor O IGUAL, sobre el valor exacto
    assert (signale == (douze['score'] >= seuil)).all()   # aquí, el redondeo no cambia ninguna decisión
    vp, fn = int((signale & est_spam).sum()), int((~signale & est_spam).sum())
    fp, vn = int((signale & ~est_spam).sum()), int((~signale & ~est_spam).sum())
    print(f'umbral {seuil}: VP = {vp}, FN = {fn}, FP = {fp}, VN = {vn} | '
          f'exhaustividad = {vp}/{vp + fn} = {vp / (vp + fn):.3f}, '            # denominador: la fila de los spams
          f'falsas alarmas = {fp}/{fp + vn} = {fp / (fp + vn):.3f}, '   # denominador: la fila de los legítimos
          f'precisión = {vp}/{vp + fp} = {vp / (vp + fp):.3f}')         # denominador: la columna de los marcados
bascule = douze['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('mensajes que cambian de 0,5 a 0,25:', list(bascule))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, seuil in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, est_spam):
        sig = q >= seuil
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGITIME,
                   edgecolor=SIGNALE if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(seuil, color=SEUIL, linewidth=2.5)
    ax.set_yticks([0, 1], ['legítimos', 'spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'umbral {seuil}', loc='left', color=SEUIL)
axes[1].set_xlabel('Probabilidad estimada (valor exacto)'); plt.tight_layout(); plt.show()

In [ ]:
# Lección 5.2 — puntuación bruta, probabilidad estimada, decisión; el caso de M3; calibración
tableau = pd.DataFrame({'message': douze['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'p arrondie': p_exact.round(2),
                        'décision (seuil 0,5)': np.where(p_exact >= 0.5, 'signalé', 'non')})
print(tableau.to_string(index=False))
print('redondeados = puntuaciones de la lección 5.1:', np.allclose(p_exact.round(2), douze['score']))
print(f"M3: z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} exactamente → "
      f"{'marcado' if p_exact[2] >= 0.5 else 'no marcado'} con el umbral 0,5")
print(f'M4: e^(-0,5) = {np.exp(-0.5):.4f}, sigmoide(0,5) = {sigmoide(0.5):.3f}; M7: sigmoide(-0,5) = {sigmoide(-0.5):.3f}')
print('umbral 0,5 sobre p̂ ⇔ umbral 0 sobre z:', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| umbral 0,25 sobre p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoide(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=SEUIL, linewidth=2)
plt.scatter(z[est_spam], p_exact[est_spam], marker='s', color=SPAM, zorder=3, label='spams')
plt.scatter(z[~est_spam], p_exact[~est_spam], marker='o', color=LEGITIME, edgecolor='gray', zorder=3, label='legítimos')
plt.xlabel('Puntuación bruta z'); plt.ylabel('Probabilidad estimada p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# Una probabilidad estimada se comprueba: lado a lado, p̂ media predicha, frecuencia observada, número de mensajes (denominador)
cal = calibration.copy()
cal['fréquence observée'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['tranche_min', 'tranche_max', 'proba_moyenne', 'fréquence observée', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"tramo 0,7–0,9: predicho {t['proba_moyenne']:.2f}, observado {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | esperado {t['proba_moyenne'] * t['messages']:.0f} spams, "
      f"desviación típica {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('diferencia máxima entre predicho y observado:', f"{np.max(np.abs(cal['proba_moyenne'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Lección 5.3 — la penalización de un spam real: −ln(p̂) (ecuación 5.5). np.log es el logaritmo NATURAL.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → penalización = {-np.log(p):.3f}')
print(f'cocientes: {np.log(0.01) / np.log(0.5):.1f} veces la versión con 0,50, '
      f'{np.log(0.01) / np.log(0.9):.1f} veces la versión con 0,90')
p_spams = p_exact[est_spam]                             # valores NO redondeados de la lección 5.2 (M4: 0,622)
penalites = -np.log(p_spams)
print('spams', list(douze.loc[est_spam, 'message']), ': penalizaciones', penalites.round(2), f'| media = {penalites.mean():.2f}')
print(f'producto de las probabilidades = {p_spams.prod():.3f}; −ln(producto) = {-np.log(p_spams.prod()):.2f} '
      f'= suma de las penalizaciones = {penalites.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLEU, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Probabilidad estimada p̂ para un spam real'); plt.ylabel('Penalización −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lección 5.4 — la pérdida de cada mensaje (ecuación 5.6) y la pérdida media (ecuación 5.7)
def perte(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln de la probabilidad dada a la respuesta correcta

p_tab = douze['score'].to_numpy()                       # valores mostrados en la tabla (a la centésima)
pertes = perte(y, p_exact)                              # pérdidas calculadas sobre los valores NO redondeados, como en el libro
print(pd.DataFrame({'message': douze['message'], 'y': y, 'p̂': p_tab,
                    'bonne réponse': np.where(y == 1, p_tab, 1 - p_tab), 'perte': pertes.round(2)}).to_string(index=False))
print(f'M4: −ln({p_exact[3]:.3f}) = {pertes[3]:.3f} ≈ {pertes[3]:.2f} (con 0,62 redondeado, obtendríamos {-np.log(0.62):.2f})')
print(f'M3: −ln(1 − 0,50) = {-np.log(0.5):.3f}; M6: −ln(1 − 0,27) = {-np.log(0.73):.3f}; M11: {-np.log(0.98):.3f}')
print(f'suma de las pérdidas redondeadas = {pertes.round(2).sum():.2f}; pérdida media = {pertes.mean():.2f} '
      f'(suma sin redondear: {pertes.sum():.3f})')

# Ajustar primero: la pérdida media en función de b (w fijo), como la curva de pérdida del capítulo 3
def perte_b(bb):
    return perte(y, sigmoide(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb}: pérdida media = {perte_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [perte_b(bb) for bb in bs]
print(f'fondo de la curva: b ≈ {bs[int(np.argmin(Lb))]:.2f}, pérdida {min(Lb):.4f} (frente a {perte_b(-4):.4f} para b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLEU, linewidth=2.2)
plt.scatter([-4], [perte_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Constante b'); plt.ylabel('Pérdida media'); plt.grid(alpha=0.4); plt.show()
print('Error al cuadrado para el spam puntuado con 0,01:', round((1 - 0.01) ** 2, 2), '; para 0,50:', (1 - 0.5) ** 2)

In [ ]:
# Lección 5.5 — comparar umbrales: alertas, falsas alarmas, no detectados, precisión, exhaustividad, F1, coste (k = 3)
lignes = []
for seuil in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= seuil                              # valores exactos
    vp, fp = int((sig & est_spam).sum()), int((sig & ~est_spam).sum())
    fn = int((~sig & est_spam).sum())
    precision = None if vp + fp == 0 else vp / (vp + fp)   # columna de los marcados vacía → indefinida
    lignes.append({'seuil': seuil, 'alertes': vp + fp, 'fausses alertes': fp, 'manqués': fn,
                   'précision': affiche(precision), 'rappel': f'{vp / (vp + fn):.2f}',
                   'F1': f'{2 * vp / (2 * vp + fp + fn):.2f}', 'coût (k = 3)': fn + 3 * fp})
print(pd.DataFrame(lignes).to_string(index=False))
print(f'F1 con el umbral 0,3: 2 × 0,8 × 1 / 1,8 = {2 * 0.8 * 1 / 1.8:.2f}; con el umbral 0,6: 1,5 / 1,75 = {1.5 / 1.75:.2f}')

# El mismo cálculo con scikit-learn, si está instalado: su convención para una precisión indefinida es 0
try:
    from sklearn.metrics import precision_score, f1_score
    predit = (p_exact >= 0.96).astype(int)
    print('scikit-learn con el umbral 0,96: precisión =', precision_score(y, predit, zero_division=0),
          '(convención zero_division=0); F1 =', f1_score(y, predit, zero_division=0))
except ImportError:
    print('scikit-learn no está instalado: este cálculo es opcional.')

## Ejercicio 5.1 — Leer un gráfico: dos filtros frente a la diagonal

*Nociones: probabilidad estimada y calibración (lección 5.2).* Lee primero la figura del libro. Filtro A: datos D5.2; filtro B: datos D5.E1.

### a) a d) Tus lecturas

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: e) frecuencias observadas del filtro B (spams / número de mensajes de cada tramo),
# después, el número de spams esperado en el tramo 0,9–1 si B estuviera calibrado (p̂ media × número de mensajes)
freq_b = None
attendu_derniere = None

In [ ]:
# Comprobación de e)
if freq_b is None or attendu_derniere is None:
    print('aún sin completar. Completa la celda anterior.')
else:
    print('frecuencias de B:', np.round(freq_b, 2), '→',
          'correcto' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'revisar')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['proba_moyenne'].iloc[-1]
    et = np.sqrt(n * q * (1 - q))                       # desviación típica del número de spams (ejercicio 4.6)
    observe = calibration_b['spams'].iloc[-1]
    print(f'tramo 0,9–1: esperado {attendu_derniere:.2f} spams, observado {observe}, desviación típica {et:.2f}, '
          f'diferencia = {(attendu_derniere - observe) / et:.0f} desviaciones típicas →',
          'correcto' if abs(attendu_derniere - 72.75) < 0.01 else 'revisar')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='calibración perfecta')
    plt.plot(calibration['proba_moyenne'], calibration['spams'] / calibration['messages'], 'o-', color=BLEU, label='filtro A')
    plt.plot(calibration_b['proba_moyenne'], freq_b, '^--', color=VERT, label='filtro B')
    plt.xlabel('Probabilidad media predicha'); plt.ylabel('Frecuencia observada'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Tu interpretación

*Tu respuesta:* …

## Ejercicio 5.2 — Tres umbrales para una asociación

*Nociones: regla de decisión y matriz de confusión (lección 5.1), exhaustividad (recall), tasa de falsas alarmas y precisión (ecuación 5.2), F1 y coste (lección 5.5).* Diez mensajes revisados por la secretaría (datos D5.E2), tres umbrales: 0,3, 0,5 y 0,7. **Las probabilidades de la tabla son los valores que hay que usar tal cual.** Haz primero las preguntas a) a f) en papel.

### a) a c) En papel

*Tus matrices y tus cálculos:* …

### b) La matriz de confusión en Python

In [ ]:
# PARA COMPLETAR: las cuatro casillas de la matriz de confusión para un umbral dado
def matrice_confusion(scores, est_spam, seuil):
    """scores y est_spam: dos listas de la misma longitud. Devuelve el diccionario
    {'VP': ..., 'FN': ..., 'FP': ..., 'VN': ...}. Regla: marcado si score >= seuil."""
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación: primero con los doce mensajes (valores EXACTOS), después con los diez mensajes del ejercicio
cas = [('lección, umbral 0,5', list(p_exact), list(est_spam), 0.5, (3, 1, 1, 7)),
       ('lección, umbral 0,25', list(p_exact), list(est_spam), 0.25, (4, 0, 2, 6)),
       ('ejercicio, umbral 0,7', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('ejercicio, umbral 0,5', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('ejercicio, umbral 0,3', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.3, (4, 1, 3, 2))]
for nom, sc, sp, seuil, attendu in cas:
    m = matrice_confusion(sc, sp, seuil)
    if m is None:
        print(f'{nom}: aún sin completar. Completa la función.')
    else:
        obtenu = (m['VP'], m['FN'], m['FP'], m['VN'])
        print(f'{nom}: VP, FN, FP, VN = {obtenu} →', 'correcto' if obtenu == attendu else 'revisar')

### c) y e) Las medidas y el F1

In [ ]:
# PARA COMPLETAR: exhaustividad, tasa de falsas alarmas, precisión (ecuación 5.2) y F1 (ecuación 5.8).
# Devuelve None para una medida cuyo denominador es cero (convención del notebook).
def mesures(m):
    rappel = None                 # PARA COMPLETAR: denominador = fila de los spams
    taux_fausses_alertes = None   # PARA COMPLETAR: denominador = fila de los mensajes legítimos
    precision = None              # PARA COMPLETAR: denominador = columna de los marcados
    f1 = None                     # PARA COMPLETAR: 2 VP / (2 VP + FP + FN)
    return rappel, taux_fausses_alertes, precision, f1

In [ ]:
# Comprobación de c) y e)
scores_e2, spams_e2 = list(dix['score']), list(dix['etiquette'] == 'spam')
if matrice_confusion(scores_e2, spams_e2, 0.5) is None:
    print('Completa primero matrice_confusion.')
elif mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Medidas: aún sin completar. Completa la función mesures.')
else:
    attendus = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for seuil, att in attendus.items():
        r, f, p, _ = mesures(matrice_confusion(scores_e2, spams_e2, seuil))
        print(f'umbral {seuil}: exhaustividad = {affiche(r)}, falsas alarmas = {affiche(f)}, precisión = {affiche(p)} →',
              'correcto' if np.allclose((r, f, p), att) else 'revisar')
    r, f, p, f1 = mesures(matrice_confusion(scores_e2, spams_e2, 0.95))
    print(f'umbral 0.95: ninguna alerta; exhaustividad = {affiche(r)}, precisión = {affiche(p)}, F1 = {affiche(f1)} →',
          'correcto' if r == 0 and p is None and f1 == 0 else 'revisar')
    f1s = [mesures(matrice_confusion(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 con los umbrales 0,7 / 0,5 / 0,3:', [round(v, 2) for v in f1s], '→',
          'correcto' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'revisar')

### f) El coste de los errores

In [ ]:
# PARA COMPLETAR: coste total = spams no detectados + k × falsas alarmas (ecuación 5.9)
def cout(m, k):
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación de f)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or cout({'FN': 0, 'FP': 0}, 3) is None:
    print('Coste: aún sin completar. Completa las funciones.')
else:
    for k, attendu in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        couts = [cout(matrice_confusion(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        meilleur = (0.7, 0.5, 0.3)[couts.index(min(couts))]
        print(f'k = {k}: costes con los umbrales 0,7 / 0,5 / 0,3 = {couts}, mejor umbral {meilleur} →',
              'correcto' if couts == attendu else 'revisar')

### g) Todos los valores del umbral

In [ ]:
# Exhaustividad, precisión y F1 para los umbrales 0, 0,01, …, 1 (precisión indefinida: sin punto)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Completa matrice_confusion y mesures para trazar las curvas.')
else:
    seuils = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[mesures(matrice_confusion(scores_e2, spams_e2, s)) for s in seuils])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(seuils, R, where='post', color=SPAM, linewidth=2, label='exhaustividad (fila de los spams)')
    plt.step(seuils, P, where='post', color=SIGNALE, linewidth=2, linestyle='--', label='precisión (columna de los marcados)')
    plt.step(seuils, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=SEUIL, linewidth=1, alpha=0.6)
    plt.xlabel('Umbral'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) a f) Tus interpretaciones

*Tu respuesta d):* …

*Tu F1 en e):* …

*Tu elección en f):* …

## Ejercicio 5.3 — Interpretar: mismas decisiones, pérdidas distintas

*Nociones: pérdida logarítmica (lecciones 5.3 y 5.4), regla de decisión (ecuación 5.1).* Datos D5.E3: estas probabilidades son los valores que hay que usar.

### a) a d) En papel

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: pérdida logarítmica media (ecuaciones 5.6 y 5.7)
def perte_moyenne(etiquettes, probas):
    """etiquettes: lista de 1 (spam) y 0 (legítimo); probas: probabilidades estimadas de ser spam."""
    return None   # PARA COMPLETAR; np.log es el logaritmo natural

In [ ]:
# Comprobación
y4 = list((quatre['etiquette'] == 'spam').astype(int))
if perte_moyenne(y4, quatre['p_filtre_A']) is None:
    print('aún sin completar. Completa la función.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for nom, probas, att in (('A', quatre['p_filtre_A'], 0.3670), ('B', quatre['p_filtre_B'], 0.0577), ('C', p_c, 1.1827)):
        v = perte_moyenne(y4, probas)
        print(f'filtro {nom}: pérdida media = {v:.4f} →', 'correcto' if abs(v - att) < 0.0001 else 'revisar')
    print('control con los doce mensajes de la lección 5.4:', round(perte_moyenne(list(y), list(p_exact)), 2))

## Ejercicio 5.4 — El coste decide el umbral

*Nociones: coste de un umbral (ecuación 5.9), F1 (ecuación 5.8).* Retomamos los doce mensajes, con sus probabilidades **exactas** `p_exact`: la regla se aplica a ellas.

### a) Tu predicción

*Si una falsa alarma cuesta cada vez más, el mejor umbral…* …

In [ ]:
# PARA COMPLETAR: coste de un umbral para los doce mensajes (valores exactos), k = precio de una falsa alarma
def cout_total(seuil, k):
    return None   # PARA COMPLETAR: spams no detectados + k × falsas alarmas; usa p_exact y est_spam

In [ ]:
# b) a e) Los mejores umbrales para cada k, después los umbrales del F1
seuils = np.round(np.arange(0, 1.001, 0.01), 2)
if cout_total(0.5, 3) is None:
    print('aún sin completar. Completa la función.')
else:
    attendus = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in attendus.items():
        c = np.array([cout_total(s, k) for s in seuils])
        meilleurs = seuils[np.isclose(c, c.min())]
        print(f'k = {k:>4}: coste mínimo {c.min():.1f}, alcanzado con los umbrales de {meilleurs.min():.2f} a {meilleurs.max():.2f} →',
              'correcto' if (meilleurs.min(), meilleurs.max()) == att else 'revisar')
        plt.step(seuils, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Umbral'); plt.ylabel('Coste (dividido por su máximo)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in seuils:
    sig = p_exact >= s
    vp, fp, fn = (sig & est_spam).sum(), (sig & ~est_spam).sum(), (~sig & est_spam).sum()
    f1.append(2 * vp / (2 * vp + fp + fn))
f1 = np.array(f1)
best = seuils[np.isclose(f1, f1.max())]
print(f'F1 máximo {f1.max():.2f}, con los umbrales de {best.min():.2f} a {best.max():.2f}')
print('M6: valor exacto', round(float(p_exact[5]), 4), '→ ¿marcado con el umbral 0,27?', bool(p_exact[5] >= 0.27))

### c) a e) Tus interpretaciones

*Tus respuestas:* …

## Ejercicio 5.5 — Predecir y luego comprobar: un redondeo al borde del umbral

*Nociones: regla de decisión (ecuación 5.1), sigmoide (ecuación 5.4), F1 (ecuación 5.8).* Mismos pesos, pero una constante b = −4,01.

### a) a c) Tus predicciones y cálculos

*Tu predicción:* …

In [ ]:
# PARA COMPLETAR: la probabilidad exacta de M3 con b = −4,01, y su decisión con el umbral 0,5
p_m3 = None            # un número (usa sigmoide)
signale_m3 = None      # True o False

In [ ]:
# Comprobación, después d) decisiones sobre los valores exactos y sobre los valores redondeados
if p_m3 is None or signale_m3 is None:
    print('aún sin completar. Completa la celda anterior.')
else:
    print(f'M3: p̂ = {p_m3:.4f} (mostrado {p_m3:.2f}), marcado: {signale_m3} →',
          'correcto' if abs(p_m3 - 0.4975) < 0.0001 and signale_m3 is False else 'revisar')
p_coll = sigmoide(X @ w - 4.01)
for seuil in (0.5, 0.25):
    exact, arrondi = p_coll >= seuil, p_coll.round(2) >= seuil
    diff = list(douze['message'][exact != arrondi])
    print(f'umbral {seuil}: decisiones distintas entre valor exacto y redondeado: {diff if diff else "ninguna"}')
sig = p_coll >= 0.5
vp, fp, fn = int((sig & est_spam).sum()), int((sig & ~est_spam).sum()), int((~sig & est_spam).sum())
print(f'colega, umbral 0,5: VP = {vp}, FN = {fn}, FP = {fp}; precisión = {vp / (vp + fp):.2f}, '
      f'exhaustividad = {vp / (vp + fn):.2f}, F1 = {2 * vp / (2 * vp + fp + fn):.2f}')

## Ejercicio 5.6 — Reto: descender por la pérdida logarítmica

*Nociones: pérdida media (ecuación 5.7), derivadas parciales y gradiente (ecuación 3.6, lección 3.4), sigmoide (ecuación 5.4).*

### a) a c) En papel

*Tus demostraciones y cálculos:* …

In [ ]:
# PARA COMPLETAR: gradiente de la pérdida media de los doce mensajes (pregunta b)
def gradient(w_, b_):
    """Devuelve (dL/dw, dL/db): dL/dw es un array de dos números."""
    return None   # PARA COMPLETAR: usa X, y y sigmoide

In [ ]:
# Comprobación, después d) 1000 pasos de gradiente (η = 0,5) desde w = (1; 0,5), b = −4
def perte_moy(w_, b_):
    p_ = sigmoide(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('aún sin completar. Completa la función.')
else:
    gw, gb = gradient(w, b)
    print(f'al principio: dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'correcto' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'revisar')
    w_, b_ = w.astype(float), float(b)
    for pas in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if pas in (1, 10, 100, 1000):
            print(f'tras {pas:>4} pasos: pérdida = {perte_moy(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('pérdida final →', 'correcto' if abs(perte_moy(w_, b_) - 0.0308) < 0.0005 else 'revisar')
# e) Pesos que separan perfectamente los doce mensajes: palabras sospechosas ≥ 3 para los spams, ≤ 2 para los legítimos
for facteur in (1, 10):
    print(f'w = (0 ; {2 * facteur}), b = {-5 * facteur}: pérdida media = {perte_moy(np.array([0, 2.0]) * facteur, -5.0 * facteur):.6f}')

### d) y e) Tus interpretaciones

*Tus respuestas:* …